In [ ]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="consensus_then_avg_margin_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [9]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [10]:
model_name = "typeform/distilbert-base-uncased-mnli"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

id2label = {int(k): v for k, v in model.config.id2label.items()}
label_lookup = {idx: str(label).lower().strip() for idx, label in id2label.items()}

def find_label_id(target):
    matches = [idx for idx, label in label_lookup.items() if label == target]
    if not matches:
        raise ValueError(f"Could not find label id for {target!r} in {label_lookup}")
    return matches[0]

contradiction_id = find_label_id("contradiction")
entailment_id = find_label_id("entailment")

print("model:", model_name)
print("id2label:", id2label)
print("contradiction_id:", contradiction_id)
print("entailment_id:", entailment_id)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

model: typeform/distilbert-base-uncased-mnli
id2label: {0: 'ENTAILMENT', 1: 'NEUTRAL', 2: 'CONTRADICTION'}
contradiction_id: 2
entailment_id: 0


In [11]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", float(y_true.mean()))


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [12]:
batch_size = 64
logits_12_parts = []
logits_21_parts = []
argmax_12_parts = []
argmax_21_parts = []

with torch.no_grad():
    for i in tqdm(range(0, len(ds), batch_size)):
        batch_s1 = sent1[i:i + batch_size]
        batch_s2 = sent2[i:i + batch_size]

        enc_12 = tokenizer(
            batch_s1,
            batch_s2,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        enc_21 = tokenizer(
            batch_s2,
            batch_s1,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )

        enc_12 = {k: v.to(device) for k, v in enc_12.items()}
        enc_21 = {k: v.to(device) for k, v in enc_21.items()}

        logits_12 = model(**enc_12).logits
        logits_21 = model(**enc_21).logits

        logits_12_parts.append(logits_12.cpu().numpy())
        logits_21_parts.append(logits_21.cpu().numpy())
        argmax_12_parts.append(torch.argmax(logits_12, dim=-1).cpu().numpy())
        argmax_21_parts.append(torch.argmax(logits_21, dim=-1).cpu().numpy())

logits_12 = np.concatenate(logits_12_parts, axis=0)
logits_21 = np.concatenate(logits_21_parts, axis=0)
argmax_12 = np.concatenate(argmax_12_parts, axis=0)
argmax_21 = np.concatenate(argmax_21_parts, axis=0)

print("done")
print("logits_12 shape:", logits_12.shape)
print("logits_21 shape:", logits_21.shape)


  0%|          | 0/7 [00:00<?, ?it/s]

done
logits_12 shape: (408, 3)
logits_21 shape: (408, 3)


In [13]:
margin_12 = logits_12[:, entailment_id] - logits_12[:, contradiction_id]
margin_21 = logits_21[:, entailment_id] - logits_21[:, contradiction_id]
avg_margin = (margin_12 + margin_21) / 2.0

gate = (argmax_12 != contradiction_id) & (argmax_21 != contradiction_id)
y_pred = ((avg_margin > 0.0) & gate).astype(int)



{'accuracy': 0.7009803921568627, 'f1': 0.769811320754717}
                precision    recall  f1-score   support

not_paraphrase       0.52      0.64      0.57       129
    paraphrase       0.81      0.73      0.77       279

      accuracy                           0.70       408
     macro avg       0.67      0.68      0.67       408
  weighted avg       0.72      0.70      0.71       408



In [ ]:
vault.create_record_list("mrpc_gate_margin_prediction", column_names=["prediction", "margin_12", "margin_21", "gate"])

for i in range(len(y_pred)):
    vault.append_record("mrpc_gate_margin_prediction", 
                        {
                            "prediction": y_pred[i],
                            "margin_12": float(margin_12[i]),
                            "margin_21": float(margin_21[i]),
                            "gate": gate[i],
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "INSERT TEXT HERE ABOUT mrpc_gate_margin_prediction"
embedding = get_embeddings(description)
vault.create_description("mrpc_gate_margin_prediction", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("mrpc_gate_margin_prediction", cat, embedding, prop)

In [ ]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])
print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


In [14]:
for i in range(5):
    print("=" * 100)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("dir_12_argmax:", int(argmax_12[i]), id2label[int(argmax_12[i])])
    print("dir_21_argmax:", int(argmax_21[i]), id2label[int(argmax_21[i])])
    print("margin_12:", float(margin_12[i]))
    print("margin_21:", float(margin_21[i]))
    print("gate:", bool(gate[i]))
    print("avg_margin:", float(avg_margin[i]))
    print("true:", int(y_true[i]))
    print("pred:", int(y_pred[i]))


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
dir_12_argmax: 0 ENTAILMENT
dir_21_argmax: 0 ENTAILMENT
margin_12: 7.967843055725098
margin_21: 6.533724784851074
gate: True
avg_margin: 7.250783920288086
true: 1
pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
dir_12_argmax: 2 CONTRADICTION
dir_21_argmax: 1 NEUTRAL
margin_12: -9.776514053344727
margin_21: -0.8201401233673096
gate: False
avg_margin: -5.2983269691467285
true: 0
pred: 0
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the ses

In [15]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 100)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("dir_12_argmax:", int(argmax_12[i]), id2label[int(argmax_12[i])])
    print("dir_21_argmax:", int(argmax_21[i]), id2label[int(argmax_21[i])])
    print("margin_12:", float(margin_12[i]))
    print("margin_21:", float(margin_21[i]))
    print("gate:", bool(gate[i]))
    print("avg_margin:", float(avg_margin[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


num_errors: 122
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
dir_12_argmax: 1 NEUTRAL
dir_21_argmax: 0 ENTAILMENT
margin_12: -0.6154601573944092
margin_21: 6.547325134277344
gate: True
avg_margin: 2.9659323692321777
true: 0 pred: 1
idx: 4
sentence1: No dates have been set for the civil or the criminal trial .
sentence2: No dates have been set for the criminal or civil cases , but Shanley has pleaded not guilty .
dir_12_argmax: 1 NEUTRAL
dir_21_argmax: 0 ENTAILMENT
margin_12: -1.2382020950317383
margin_21: 7.657938003540039
gate: True
avg_margin: 3.2098679542541504
true: 0 pred: 1
idx: 5
sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
sentence2: It has also said it would review all

In [16]:
vault.create_record_list("consensus_then_avg_margin_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("consensus_then_avg_margin_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "mrpc_gate_margin_prediction": [0, len(ds)]
                    })

summary

description = "INSERT TEXT HERE ABOUT consensus_then_avg_margin_mrpc_summary"
embedding = get_embeddings(description)
vault.create_description("consensus_then_avg_margin_mrpc_summary", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("consensus_then_avg_margin_mrpc_summary", cat, embedding, prop)



{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'typeform/distilbert-base-uncased-mnli',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.7009803921568627,
 'f1': 0.769811320754717,
 'method': 'bidirectional_mnli_contradiction_free_consensus_plus_average_margin'}

In [ ]:
description = "INSERT TEXT HERE ABOUT consensus_then_avg_margin_mrpc process/notebook" # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("consensus_then_avg_margin_mrpc", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("consensus_then_avg_margin_mrpc", cat, embedding, prop)